# results.jsonl 시각화

`batch_move_estimate.py`가 생성한 `results.jsonl`을 읽고, 선택한 지표를 케이스별 그래프로 확인합니다.
GT 값을 Python 리스트 또는 딕셔너리로 넣으면 예측값과 함께 비교 플롯을 그립니다.


## 실행 방법

이 노트북은 위 셀에서 정의한 함수들을 아래 셀들이 재사용합니다.
중간 셀만 단독 실행하면 `NameError: load_results is not defined` 같은 오류가 날 수 있습니다.

처음 열었거나 커널을 재시작했다면 `Run All`을 실행하거나, 맨 위 import 셀부터 순서대로 실행하세요.


In [ ]:
from __future__ import annotations

import json
import math
import csv
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt


## 설정

- `RESULTS_PATH`: 시각화할 결과 JSONL 경로
- `METRIC`: 그래프로 볼 지표 이름
- `GT_VALUES`: GT가 있으면 리스트 또는 `{case_id: value}` 딕셔너리로 입력


In [ ]:
RESULTS_PATH = Path('results.jsonl')
RESULTS_WITH_CSV_PATH = Path('results_with_csv.jsonl')
CASES_PATH = Path('cases.jsonl')
CASES_WITH_CSV_PATH = Path('cases_with_csv.jsonl')

# 이 노트북의 메인 비교 그래프는 volume_m3 전용입니다.
METRIC = 'volume_m3'

# GT 입력 방법 1: results.jsonl/cases.jsonl의 case_id 순서와 같은 리스트
GT_VALUES = [5, 2.5, 3, 3.5, 5, 2, 2.5, 5, 1, 7.5, 6, 6, 6, 5, 2, 5, 5, 1, 3.5]  # 예: [12.5, 18.0, 7.2]

# GT 입력 방법 2: case_id 기준 딕셔너리
# GT_VALUES = {'anon_0002': 12.5, 'anon_0003': 18.0}


## 결과 파일 로드 함수

`results.jsonl`과 `results_with_csv.jsonl`을 읽기 위한 공통 함수입니다.


In [ ]:
def load_results(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row['_line_no'] = line_no
            rows.append(row)
    return rows


def successful_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [row for row in rows if row.get('ok') and isinstance(row.get('result'), dict)]


In [ ]:
rows = load_results(RESULTS_PATH)
ok_rows = successful_rows(rows)
failed_rows = [row for row in rows if not row.get('ok')]

rows_with_csv = load_results(RESULTS_WITH_CSV_PATH) if RESULTS_WITH_CSV_PATH.exists() else []
ok_rows_with_csv = successful_rows(rows_with_csv)
failed_rows_with_csv = [row for row in rows_with_csv if not row.get('ok')]

print(f'base results total rows: {len(rows)}')
print(f'base results ok rows: {len(ok_rows)}')
print(f'VLM-filtered total rows: {len(rows_with_csv)}')
print(f'VLM-filtered ok rows: {len(ok_rows_with_csv)}')
print(f'VLM-filtered failed rows: {len(failed_rows_with_csv)}')
if failed_rows_with_csv:
    print('VLM-filtered failed case ids:', [row.get('case_id') for row in failed_rows_with_csv])


## 지표 계산 함수

필요한 지표가 있으면 `metric_value()`에 분기를 추가하면 됩니다.


In [ ]:
def metric_value(row: dict[str, Any], metric: str) -> float | None:
    result = row.get('result') or {}
    lines = result.get('lines') or []
    photo_items = result.get('photo_items') or []

    if metric in result:
        value = result.get(metric)
    elif metric == 'line_count':
        value = len(lines)
    elif metric == 'photo_item_count':
        value = len(photo_items)
    elif metric == 'total_qty':
        value = sum(int(item.get('qty', 1) or 1) for item in lines if isinstance(item, dict))
    elif metric == 'avg_item_volume_m3':
        volumes = []
        for item in lines:
            if not isinstance(item, dict):
                continue
            try:
                volumes.append(float(item.get('estimated_volume_m3')))
            except (TypeError, ValueError):
                pass
        value = sum(volumes) / len(volumes) if volumes else None
    else:
        quote = result.get('quote') if isinstance(result.get('quote'), dict) else {}
        value = quote.get(metric)

    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def available_basic_metrics(rows: list[dict[str, Any]]) -> list[str]:
    metrics = {'line_count', 'photo_item_count', 'total_qty', 'avg_item_volume_m3'}
    for row in rows:
        result = row.get('result') or {}
        quote = result.get('quote') if isinstance(result.get('quote'), dict) else {}
        for key, value in {**result, **quote}.items():
            if isinstance(value, (int, float)):
                metrics.add(key)
    return sorted(metrics)


print('available metrics:')
for name in available_basic_metrics(ok_rows):
    print('-', name)


## 선택 지표 데이터 만들기


In [ ]:
def values_by_case(rows: list[dict[str, Any]], metric: str) -> dict[str, float]:
    values: dict[str, float] = {}
    for row in rows:
        value = metric_value(row, metric)
        if value is None:
            continue
        values[str(row.get('case_id'))] = value
    return values


with_csv_by_case = values_by_case(ok_rows_with_csv, 'volume_m3')

# 그래프 축은 GT와 CSV 합계가 있는 전체 case_id 순서를 기준으로 둡니다.
case_ids = [str(row.get('case_id')) for row in ok_rows]
with_csv_values = [with_csv_by_case.get(case_id) for case_id in case_ids]

print('metric: volume_m3')
print(f'case axis count: {len(case_ids)}')
print(f'VLM-filtered matched cases: {sum(v is not None for v in with_csv_values)}')
list(zip(case_ids, with_csv_values))[:5]


## CSV volume_m3 합계 로드

`cases_with_csv.jsonl`의 `quote_csv_path`를 읽어 각 CSV의 `volume_m3` 컬럼을 단순 합산합니다.
이 값은 `METRIC = 'volume_m3'`일 때 예측 부피와 함께 표시됩니다.


In [ ]:
def load_case_csv_paths(path: Path) -> dict[str, Path]:
    if not path.exists():
        return {}
    base_dir = path.resolve().parent
    mapping: dict[str, Path] = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            case_id = str(row.get('case_id') or '')
            csv_path = row.get('quote_csv_path')
            if not case_id or not csv_path:
                continue
            path_obj = Path(str(csv_path)).expanduser()
            if not path_obj.is_absolute():
                path_obj = base_dir / path_obj
            mapping[case_id] = path_obj
    return mapping


def csv_volume_m3_sum(csv_path: Path) -> float | None:
    if not csv_path.exists():
        return None
    total = 0.0
    found = False
    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                total += float(row.get('volume_m3') or 0)
                found = True
            except (TypeError, ValueError):
                pass
    return total if found else None


case_csv_paths = load_case_csv_paths(CASES_WITH_CSV_PATH)
csv_volume_by_case = {
    case_id: value
    for case_id, csv_path in case_csv_paths.items()
    if (value := csv_volume_m3_sum(csv_path)) is not None
}

print(f'CSV paths loaded: {len(case_csv_paths)}')
print(f'CSV volume sums loaded: {len(csv_volume_by_case)}')
list(csv_volume_by_case.items())[:5]


## 공용 그래프 함수

앞으로 다른 지표 그래프를 추가할 때도 `plot_case_lines()`에 series를 넘겨 재사용할 수 있습니다.


In [ ]:
def plot_case_lines(case_ids: list[str], series: dict[str, list[float | None]], ylabel: str, title: str) -> None:
    x = list(range(len(case_ids)))
    plt.figure(figsize=(max(10, len(case_ids) * 0.55), 5))
    for label, values in series.items():
        y = [math.nan if value is None else float(value) for value in values]
        plt.plot(x, y, marker='o', linewidth=2, label=label)
    plt.xticks(x, case_ids, rotation=60, ha='right')
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(axis='both', alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 정규화 함수

각 시리즈를 개별적으로 min-max 정규화해 0~1 범위에서 상대적인 변화 패턴을 비교합니다.


In [ ]:
def minmax_normalize(values: list[float | None]) -> list[float | None]:
    valid = [float(value) for value in values if value is not None]
    if not valid:
        return [None for _ in values]
    min_v = min(valid)
    max_v = max(valid)
    if max_v == min_v:
        return [0.5 if value is not None else None for value in values]
    return [None if value is None else (float(value) - min_v) / (max_v - min_v) for value in values]


def normalize_series_minmax(series: dict[str, list[float | None]]) -> dict[str, list[float | None]]:
    return {label: minmax_normalize(values) for label, values in series.items()}


## GT 정렬 함수

`GT_VALUES`를 현재 그래프의 `case_ids` 순서에 맞춥니다.


In [ ]:
def align_gt(case_ids: list[str], gt_values: list[float] | dict[str, float]) -> list[float | None]:
    if isinstance(gt_values, dict):
        return [gt_values.get(case_id) for case_id in case_ids]
    if isinstance(gt_values, list):
        aligned = list(gt_values[:len(case_ids)])
        aligned.extend([None] * (len(case_ids) - len(aligned)))
        return aligned
    return [None] * len(case_ids)


## volume_m3 통합 비교 플롯

CSV 단순 합계(`estimated`), VLM 필터링 결과(`VLM-filtered`), GT를 한 그래프에 표시합니다.


In [ ]:
series: dict[str, list[float | None]] = {}

csv_sum_values = [csv_volume_by_case.get(case_id) for case_id in case_ids]
if any(value is not None for value in csv_sum_values):
    series['estimated'] = csv_sum_values
    missing = [case_id for case_id, value in zip(case_ids, csv_sum_values) if value is None]
    if missing:
        print(f'estimated CSV volume_m3 sum is missing for {len(missing)} cases:', missing)
else:
    print('estimated CSV volume_m3 sum has no matched cases.')

if any(value is not None for value in with_csv_values):
    series['VLM-filtered'] = with_csv_values
    missing = [case_id for case_id, value in zip(case_ids, with_csv_values) if value is None]
    if missing:
        print(f'VLM-filtered result is missing for {len(missing)} cases:', missing)
else:
    print('VLM-filtered result has no matched cases.')

gt_values = align_gt(case_ids, GT_VALUES)
if any(value is not None for value in gt_values):
    series['GT'] = gt_values
    missing = [case_id for case_id, value in zip(case_ids, gt_values) if value is None]
    if missing:
        print(f'GT is missing for {len(missing)} cases:', missing)

plot_case_lines(
    case_ids,
    series,
    ylabel='volume_m3',
    title='volume_m3 comparison: estimated vs VLM-filtered vs GT',
)

normalized_series = normalize_series_minmax(series)
plot_case_lines(
    case_ids,
    normalized_series,
    ylabel='normalized volume_m3 (min-max per series)',
    title='normalized volume_m3 comparison (min-max per series)',
)


## GT 오차 요약

통합 그래프에 표시된 GT를 기준으로 `estimated`와 `VLM-filtered` 결과의 MAE/MAPE를 계산합니다.


In [ ]:
gt_values = align_gt(case_ids, GT_VALUES)

def error_summary(label: str, predicted: list[float | None], gt_values: list[float | None]) -> None:
    pairs = [(p, g) for p, g in zip(predicted, gt_values) if p is not None and g is not None]
    if not pairs:
        print(f'{label}: GT와 매칭되는 값이 없습니다.')
        return
    errors = [float(p) - float(g) for p, g in pairs]
    abs_errors = [abs(e) for e in errors]
    mae = sum(abs_errors) / len(abs_errors)
    mape_values = [abs(float(p) - float(g)) / abs(float(g)) * 100 for p, g in pairs if float(g) != 0]
    mape = sum(mape_values) / len(mape_values) if mape_values else None
    print(f'{label}: matched={len(pairs)}, MAE={mae:.4f}' + (f', MAPE={mape:.2f}%' if mape is not None else ''))


error_summary('estimated', csv_sum_values, gt_values)
error_summary('VLM-filtered', with_csv_values, gt_values)


## 품목 수와 부피 관계 보기

선택 지표와 별개로, 최종 품목 줄 수와 총 부피의 관계를 빠르게 확인합니다.


In [ ]:
line_counts = [metric_value(row, 'line_count') for row in ok_rows]
volumes = [metric_value(row, 'volume_m3') for row in ok_rows]
scatter_case_ids = [str(row.get('case_id')) for row in ok_rows]

points = [
    (case_id, lc, vol)
    for case_id, lc, vol in zip(scatter_case_ids, line_counts, volumes)
    if lc is not None and vol is not None
]

if points:
    plt.figure(figsize=(7, 5))
    plt.scatter([p[1] for p in points], [p[2] for p in points])
    for case_id, lc, vol in points:
        plt.annotate(case_id, (lc, vol), fontsize=8, alpha=0.75)
    plt.xlabel('line_count')
    plt.ylabel('volume_m3')
    plt.title('line_count vs volume_m3')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
